# UMTAM Curvature-Aware Pruning: QQP (Quora Question Pairs)

**Task:** Binary classification - Are two questions semantically equivalent?

**Dataset:** QQP (Quora Question Pairs)
- Train: 363,846 question pairs
- Validation: 40,430 question pairs
- Metrics: Accuracy and F1

**Model:** BERT-base-uncased (110M parameters)

**Hypothesis:** Like MRPC (paraphrase detection), QQP should show strong curvature advantage since both are semantic similarity tasks. The much larger dataset may provide even better curvature estimates.

**Expected Runtime:** ~2-3 hours (large dataset)

---
## 📦 Setup

In [ ]:
# Install packages
!pip install -q transformers datasets torch evaluate matplotlib seaborn
print("✓ Packages installed")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import evaluate
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"✓ Device: {device}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## ⚙️ Configuration

In [ ]:
CONFIG = {
    # Model
    'model_name': 'bert-base-uncased',
    'num_labels': 2,
    
    # Training
    'max_length': 128,
    'batch_size': 32,  # Can handle larger batches with A100
    'num_epochs': 3,   # 3 epochs sufficient for large dataset
    'lr': 2e-5,
    
    # Dataset (QQP is very large - can subsample for faster experiments)
    'use_full_dataset': True,  # Set to False to use subset
    'max_train_samples': 50000,  # Only used if use_full_dataset=False
    'max_eval_samples': 10000,   # Only used if use_full_dataset=False
    
    # Pruning
    'densities': [1, 5, 10, 20, 40, 60, 90],
    
    # Metrics
    'primary_metric': 'f1',  # F1 is standard for QQP (like MRPC)
    
    # Other
    'seed': 42,
}

print("✓ Configuration:")
print(f"  Model: {CONFIG['model_name']}")
print(f"  Primary metric: {CONFIG['primary_metric']}")
print(f"  Densities: {CONFIG['densities']}")
print(f"  Epochs: {CONFIG['num_epochs']}")
print(f"  Full dataset: {CONFIG['use_full_dataset']}")
if not CONFIG['use_full_dataset']:
    print(f"  Train samples: {CONFIG['max_train_samples']}")
    print(f"  Eval samples: {CONFIG['max_eval_samples']}")

---
## 📚 Load Dataset

In [ ]:
print("Loading QQP dataset...")
print("⚠️  This is a large dataset - may take a few minutes to download and process")

# Load tokenizer and dataset
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
dataset = load_dataset('glue', 'qqp')

print(f"\nDataset sizes:")
print(f"  Train: {len(dataset['train']):,} question pairs")
print(f"  Validation: {len(dataset['validation']):,} question pairs")

# Tokenize
def tokenize_function(examples):
    result = tokenizer(
        examples['question1'],
        examples['question2'],
        padding='max_length',
        truncation=True,
        max_length=CONFIG['max_length']
    )
    result['labels'] = examples['label']
    return result

print("\nTokenizing dataset...")
tokenized = dataset.map(tokenize_function, batched=True)

# Prepare train and eval datasets
train_dataset = tokenized['train'].shuffle(seed=CONFIG['seed'])
eval_dataset = tokenized['validation']

# Optionally subsample for faster experiments
if not CONFIG['use_full_dataset']:
    print(f"\n⚠️  Using subset of data for faster experimentation")
    train_dataset = train_dataset.select(range(CONFIG['max_train_samples']))
    eval_dataset = eval_dataset.select(range(CONFIG['max_eval_samples']))
else:
    print(f"\n✓ Using full dataset (recommended for publication)")

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
eval_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f"\n✓ Dataset prepared")
print(f"  Train: {len(train_dataset):,} samples")
print(f"  Eval: {len(eval_dataset):,} samples")
print(f"  Training batches: {len(train_dataset) // CONFIG['batch_size']:,}")

---
## 🏋️ Training Function

In [ ]:
def train_and_collect_curvature(model, train_dataset):
    """
    Train model with AdamW and collect curvature statistics.
    Returns: trained model, initial weights, curvature-aware saliency scores
    """
    model.to(device)
    model.train()
    
    optimizer = AdamW(model.parameters(), lr=CONFIG['lr'])
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
    
    # Store initial weights (w0)
    print("\nStoring initial weights...")
    w0_dict = {}
    for name, param in model.named_parameters():
        if len(param.shape) == 2:  # Only weight matrices
            w0_dict[name] = param.data.clone().cpu()
    
    print(f"Stored initial weights for {len(w0_dict)} layers")
    
    # Initialize curvature tracking (factorized: R for rows, C for columns)
    curvature_dict = {}
    for name, param in model.named_parameters():
        if len(param.shape) == 2:
            m, n = param.shape
            curvature_dict[name] = {
                'R': torch.zeros(m, device=device),  # Row-wise curvature
                'C': torch.zeros(n, device=device),  # Column-wise curvature
            }
    
    print(f"Tracking curvature for {len(curvature_dict)} layers")
    
    # Training loop
    print(f"\nTraining for {CONFIG['num_epochs']} epochs...")
    for epoch in range(CONFIG['num_epochs']):
        total_loss = 0
        num_batches = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']}")
        
        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            
            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            
            # Accumulate gradient statistics (curvature proxy)
            with torch.no_grad():
                for name, param in model.named_parameters():
                    if name in curvature_dict and param.grad is not None:
                        grad_sq = param.grad ** 2
                        curvature_dict[name]['R'] += grad_sq.sum(dim=1)
                        curvature_dict[name]['C'] += grad_sq.sum(dim=0)
            
            optimizer.step()
            total_loss += loss.item()
            num_batches += 1
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / num_batches
        print(f"  Epoch {epoch+1} average loss: {avg_loss:.4f}")
    
    # Compute curvature-aware saliency scores
    print("\nComputing saliency scores...")
    saliency_dict = {}
    
    for name in tqdm(curvature_dict.keys(), desc="Computing saliency"):
        param = dict(model.named_parameters())[name]
        w0 = w0_dict[name].to(device)
        
        # Parameter change: (w - w0)²
        param_change = (param.data - w0) ** 2
        
        # Curvature approximation: sqrt(R ⊗ C)
        R = curvature_dict[name]['R'].clamp(min=1e-8)
        C = curvature_dict[name]['C'].clamp(min=1e-8)
        curvature_approx = torch.sqrt(R.unsqueeze(1) * C.unsqueeze(0))
        
        # Saliency = parameter_change × curvature
        saliency_dict[name] = (param_change * curvature_approx).cpu()
    
    print(f"✓ Training complete. Computed saliency for {len(saliency_dict)} layers.")
    return model, w0_dict, saliency_dict

---
## 🔪 Pruning Functions

In [ ]:
def apply_pruning_inplace(model, w0_dict, saliency_dict, density_pct, use_curvature=True):
    """
    Apply Fast Fisher Grafting in-place:
    - Keep top-saliency parameters
    - Revert low-saliency parameters to initialization (w0)
    
    If use_curvature=False, uses magnitude-only saliency: (w - w0)²
    """
    # Collect all saliency scores
    all_scores = []
    for name in saliency_dict.keys():
        if use_curvature:
            # Full curvature-aware saliency
            scores = saliency_dict[name].flatten()
        else:
            # Magnitude-only: just parameter change
            param = dict(model.named_parameters())[name]
            w0 = w0_dict[name].to(device)
            scores = ((param.data - w0) ** 2).cpu().flatten()
        
        all_scores.append(scores)
    
    all_scores = torch.cat(all_scores)
    
    # Determine threshold for keeping top density_pct
    k = int(len(all_scores) * (density_pct / 100.0))
    threshold = torch.kthvalue(all_scores, len(all_scores) - k).values
    
    # Apply grafting
    total_params = 0
    kept_params = 0
    
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in saliency_dict:
                w0 = w0_dict[name].to(device)
                
                if use_curvature:
                    scores = saliency_dict[name].to(device)
                else:
                    scores = (param.data - w0) ** 2
                
                # Create mask: 1 if keep, 0 if revert to w0
                mask = (scores > threshold).float()
                
                # Graft: param = mask * param + (1-mask) * w0
                param.data = mask * param.data + (1 - mask) * w0
                
                total_params += mask.numel()
                kept_params += mask.sum().item()
    
    actual_density = 100.0 * kept_params / total_params if total_params > 0 else 0
    print(f"  Pruned to {actual_density:.1f}% density (target: {density_pct}%)")

def restore_weights(model, w_trained_dict):
    """Restore model to trained state (before pruning)"""
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in w_trained_dict:
                param.data.copy_(w_trained_dict[name].to(device))

print("✓ Pruning functions defined")

---
## 📊 Evaluation Function

In [ ]:
def evaluate_model(model, eval_dataset):
    """Evaluate model on validation set (returns both accuracy and F1)"""
    model.eval()
    model.to(device)
    
    eval_loader = DataLoader(eval_dataset, batch_size=CONFIG['batch_size'])
    metric = evaluate.load('glue', 'qqp')
    
    with torch.no_grad():
        for batch in tqdm(eval_loader, desc="Evaluating", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            predictions = torch.argmax(outputs.logits, dim=-1)
            metric.add_batch(predictions=predictions, references=batch['labels'])
    
    result = metric.compute()
    return result

print("✓ Evaluation function defined")

---
## 🚀 Run Experiment

In [ ]:
import time

start_time = time.time()

print("="*70)
print("QQP EXPERIMENT: Curvature-Aware Pruning")
print("="*70)

# Load model
print("\nLoading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=CONFIG['num_labels']
)
print(f"✓ Loaded {CONFIG['model_name']}")

# Train and collect curvature
model, w0_dict, saliency_dict = train_and_collect_curvature(model, train_dataset)

# Save trained weights (for restoring after pruning)
print("\nSaving trained weights...")
w_trained_dict = {}
for name, param in model.named_parameters():
    if name in w0_dict:
        w_trained_dict[name] = param.data.clone().cpu()
print("✓ Saved trained weights")

# Evaluate full model
print("\n" + "="*70)
print("Evaluating full model...")
print("="*70)
full_result = evaluate_model(model, eval_dataset)
full_perf = full_result[CONFIG['primary_metric']]
print(f"\nFull model results:")
print(f"  Accuracy: {full_result['accuracy']:.4f}")
print(f"  F1: {full_result['f1']:.4f}")
print(f"  Primary metric ({CONFIG['primary_metric']}): {full_perf:.4f}")

# Test pruning at different densities
results = {
    'umtam': [],
    'magnitude': [],
    'umtam_accuracy': [],
    'magnitude_accuracy': [],
    'densities': CONFIG['densities'],
    'full_performance': full_perf,
    'full_accuracy': full_result['accuracy'],
}

print("\n" + "="*70)
print("Testing pruning at different densities...")
print("="*70)

for density in CONFIG['densities']:
    print(f"\n{'─'*70}")
    print(f"Density: {density}%")
    print(f"{'─'*70}")
    
    # Test curvature-aware (UMTAM)
    print("\n[1/2] Curvature-aware (UMTAM)...")
    apply_pruning_inplace(model, w0_dict, saliency_dict, density, use_curvature=True)
    result_umtam = evaluate_model(model, eval_dataset)
    perf_umtam = result_umtam[CONFIG['primary_metric']]
    results['umtam'].append(perf_umtam)
    results['umtam_accuracy'].append(result_umtam['accuracy'])
    print(f"  → Accuracy: {result_umtam['accuracy']:.4f}")
    print(f"  → F1: {result_umtam['f1']:.4f}")
    
    # Restore to trained state
    restore_weights(model, w_trained_dict)
    
    # Test magnitude-only
    print("\n[2/2] Magnitude-only...")
    apply_pruning_inplace(model, w0_dict, saliency_dict, density, use_curvature=False)
    result_mag = evaluate_model(model, eval_dataset)
    perf_mag = result_mag[CONFIG['primary_metric']]
    results['magnitude'].append(perf_mag)
    results['magnitude_accuracy'].append(result_mag['accuracy'])
    print(f"  → Accuracy: {result_mag['accuracy']:.4f}")
    print(f"  → F1: {result_mag['f1']:.4f}")
    
    # Compare
    diff = perf_umtam - perf_mag
    if perf_mag > 0.01:
        improvement = (diff / perf_mag) * 100
        print(f"\n  📊 F1 Comparison: {diff:+.4f} ({improvement:+.1f}%)")
        if diff > 0.01:
            print(f"  ✓ UMTAM better")
        elif diff < -0.01:
            print(f"  ✗ Magnitude better")
    
    # Restore for next iteration
    restore_weights(model, w_trained_dict)

elapsed_time = time.time() - start_time

print("\n" + "="*70)
print("✓ EXPERIMENT COMPLETE!")
print(f"Total time: {elapsed_time/60:.1f} minutes")
print("="*70)

---
## 📊 Results & Visualization

In [ ]:
# Print results table
print("\n" + "="*70)
print("RESULTS SUMMARY: QQP")
print("="*70)
print(f"\nFull model F1: {results['full_performance']:.4f}")
print(f"Full model Accuracy: {results['full_accuracy']:.4f}")
print("\n" + "─"*70)
print("F1 SCORES:")
print("─"*70)
print("Density |   UMTAM   | Magnitude | Difference | Improvement")
print("─" * 70)

for i, density in enumerate(results['densities']):
    umtam = results['umtam'][i]
    mag = results['magnitude'][i]
    diff = umtam - mag
    
    if mag > 0.01:
        pct = (diff / mag) * 100
    else:
        pct = 0
    
    symbol = "✓" if diff > 0.01 else "✗" if diff < -0.01 else "≈"
    print(f"{density:>3}%    | {umtam:>8.4f}  | {mag:>8.4f}  | {diff:>+9.4f} | {pct:>+7.1f}%  {symbol}")

print("\n" + "─"*70)
print("ACCURACY:")
print("─"*70)
print("Density |   UMTAM   | Magnitude | Difference | Improvement")
print("─" * 70)

for i, density in enumerate(results['densities']):
    umtam = results['umtam_accuracy'][i]
    mag = results['magnitude_accuracy'][i]
    diff = umtam - mag
    
    if mag > 0.01:
        pct = (diff / mag) * 100
    else:
        pct = 0
    
    symbol = "✓" if diff > 0.01 else "✗" if diff < -0.01 else "≈"
    print(f"{density:>3}%    | {umtam:>8.4f}  | {mag:>8.4f}  | {diff:>+9.4f} | {pct:>+7.1f}%  {symbol}")

# Calculate average improvement at low densities
low_density_improvements = []
for i in range(min(3, len(results['densities']))):
    if results['magnitude'][i] > 0.01:
        imp = (results['umtam'][i] - results['magnitude'][i]) / results['magnitude'][i] * 100
        low_density_improvements.append(imp)

if low_density_improvements:
    avg_imp = np.mean(low_density_improvements)
    print(f"\nAverage F1 improvement at 1-10% density: {avg_imp:+.2f}%")
    if avg_imp > 50:
        print("✓✓✓ OUTSTANDING - Exceptional curvature advantage!")
    elif avg_imp > 20:
        print("✓✓ EXCELLENT - Strong curvature advantage!")
    elif avg_imp > 10:
        print("✓ STRONG - Clear curvature benefit")
    elif avg_imp > 5:
        print("✓ GOOD - Moderate improvement")

In [ ]:
# Plot results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# F1 plot
ax1 = axes[0]
ax1.plot(results['densities'], results['umtam'], 'o-', 
         label='Curvature-aware (UMTAM)', 
         linewidth=2.5, markersize=10, color='#2E86AB')
ax1.plot(results['densities'], results['magnitude'], 's--', 
         label='Magnitude-only', 
         linewidth=2.5, markersize=10, color='#A23B72')
ax1.axhline(y=results['full_performance'], color='gray', 
            linestyle=':', label='Full Model', linewidth=2)

# Annotate improvements at low densities
for i in [0, 1, 2]:  # 1%, 5%, 10%
    if i < len(results['densities']) and results['magnitude'][i] > 0.01:
        improvement = (results['umtam'][i] - results['magnitude'][i]) / results['magnitude'][i] * 100
        if abs(improvement) > 5:
            sign = '+' if improvement > 0 else ''
            ax1.text(results['densities'][i], results['umtam'][i], 
                    f'{sign}{improvement:.0f}%', 
                    fontsize=10, ha='center', va='bottom', fontweight='bold')

ax1.set_xlabel('Density (%)', fontsize=13, fontweight='bold')
ax1.set_ylabel('F1 Score', fontsize=13, fontweight='bold')
ax1.set_title('QQP: F1 Score', fontsize=15, fontweight='bold')
ax1.legend(fontsize=11, loc='best')
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log')

# Accuracy plot
ax2 = axes[1]
ax2.plot(results['densities'], results['umtam_accuracy'], 'o-', 
         label='Curvature-aware (UMTAM)', 
         linewidth=2.5, markersize=10, color='#2E86AB')
ax2.plot(results['densities'], results['magnitude_accuracy'], 's--', 
         label='Magnitude-only', 
         linewidth=2.5, markersize=10, color='#A23B72')
ax2.axhline(y=results['full_accuracy'], color='gray', 
            linestyle=':', label='Full Model', linewidth=2)

# Annotate improvements
for i in [0, 1, 2]:
    if i < len(results['densities']) and results['magnitude_accuracy'][i] > 0.01:
        improvement = (results['umtam_accuracy'][i] - results['magnitude_accuracy'][i]) / results['magnitude_accuracy'][i] * 100
        if abs(improvement) > 2:
            sign = '+' if improvement > 0 else ''
            ax2.text(results['densities'][i], results['umtam_accuracy'][i], 
                    f'{sign}{improvement:.0f}%', 
                    fontsize=10, ha='center', va='bottom', fontweight='bold')

ax2.set_xlabel('Density (%)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Accuracy', fontsize=13, fontweight='bold')
ax2.set_title('QQP: Accuracy', fontsize=15, fontweight='bold')
ax2.legend(fontsize=11, loc='best')
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log')

plt.tight_layout()
plt.savefig('qqp_pruning_results.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: qqp_pruning_results.png")
plt.show()

# Save results as JSON
with open('qqp_pruning_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("✓ Saved: qqp_pruning_results.json")

print("\n" + "="*70)
print("All files saved. Experiment complete!")
print("="*70)